In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
del_path = os.path.join(path, 'Q1_data.csv')
df_del = pd.read_csv(del_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_del.shape}")
df_del.head()

In [ ]:
# Task 3: Write your code here:
df_del.info()

In [ ]:
# Task 4: Write your code here:
df_del.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_del['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time')
plt.xlabel('x')
plt.ylabel('how long')
plt.show()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_del)
df_clean = df_del.copy()
df_clean = df_clean.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here: #the missing are objects(Weather Traffic_Level Time_of_Day) (Courier_Experience_yrs)float
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna(df_clean['Traffic_Level'].mode()[0])
df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])
df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0])
df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].median())
df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mode()[0])

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")
# its good , hopefully<;

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
def mean_Absolute_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum(abs(y_hat - y))
n_splits = 5  # K=5 Folds
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

In [ ]:
# Task 1: Write your code here:
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MAE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: